First, we will be testing to see if we can successfully extract data from the API using the example file from MET

In [ ]:
# Libraries needed (pandas is not standard and must be installed in Python)
import requests
import pandas as pd

# Insert your own client ID here
client_id = 'client-id'

# Define endpoint and parameters
endpoint = 'https://frost.met.no/observations/v0.jsonld'
parameters = {
    'sources': 'SN18700,SN90450',
    'elements': 'mean(air_temperature P1D),sum(precipitation_amount P1D),mean(wind_speed P1D)',
    'referencetime': '2010-04-01/2010-04-03',
}

# Issue an HTTP GET request
r = requests.get(endpoint, parameters, auth=(client_id,''))

# Extract JSON data
json = r.json()

# Check if the request worked, print out any errors
if r.status_code == 200:
    data = json['data']
    print('Data retrieved from frost.met.no!')
else:
    print('Error! Returned status code %s' % r.status_code)
    print('Message: %s' % json['error']['message'])
    print('Reason: %s' % json['error']['reason'])

# Outdated code segment, append-method removed in newer versions of pandas
""" This will return a Dataframe with all of the observations in a table format
df = pd.DataFrame()
for i in range(len(data)):
    row = pd.DataFrame(data[i]['observations'])
    row['referenceTime'] = data[i]['referenceTime']
    row['sourceId'] = data[i]['sourceId']
    df = df.append(row)
df = df.reset_index()
"""

# Replace with
frames = []
for item in data:
    row = pd.DataFrame(item["observations"])
    row["referenceTime"] = item["referenceTime"]
    row["sourceId"] = item["sourceId"]
    frames.append(row)
df = pd.concat(frames, ignore_index=True)

# These additional columns will be kept
columns = ['sourceId','referenceTime','elementId','value','unit','timeOffset']
df2 = df[columns].copy()
# Convert the time value to something Python understands
df2['referenceTime'] = pd.to_datetime(df2['referenceTime'])

# Preview the result
df2.head()

Data retrieved from frost.met.no!


,sourceId,referenceTime,elementId,value,unit,timeOffset
0,SN18700:0,2010-04-01 00:00:00+00:00,mean(air_temperature P1D),3.2,degC,PT0H
1,SN18700:0,2010-04-01 00:00:00+00:00,mean(air_temperature P1D),3.0,degC,PT6H
2,SN18700:0,2010-04-01 00:00:00+00:00,sum(precipitation_amount P1D),13.5,mm,PT18H
3,SN18700:0,2010-04-01 00:00:00+00:00,sum(precipitation_amount P1D),29.0,mm,PT6H
4,SN18700:0,2010-04-01 00:00:00+00:00,mean(wind_speed P1D),1.7,m/s,PT0H
